# scoring_model 검증 노트북 (Hold-out + Good/Bad 사례 테스트)

**목적**: 500개(정제된 기준 동작)로 학습한 Mahalanobis 기준 모델이 실제로
- 처음 보는 "정상" rep에는 높은 점수를,
- 의도적으로 폼이 잘못된 rep에는 낮은 점수를

주는지 검증합니다.

## 폴더 규칙
- `data/raw/` : 기준(reference) 동작 500개. `{exercise}_{video_id}_keypoints.npy`
- `data/raw_test/` : 검증용 동작 20개 (좋은 예시 10 + 나쁜 예시 10).
  **파일명에 good/bad를 표시**해주세요: 예) `squat_good01_keypoints.npy`, `squat_bad03_keypoints.npy`
  (video_id가 "good"/"bad"로 시작하는지로 라벨을 자동 판별합니다)

## ⚠️ 반드시 지켜야 할 것
`data/raw_test/`의 "좋은 예시"는 **`data/raw/`의 500개와 겹치지 않는, 별도로 촬영한 영상**이어야
합니다. 학습에 쓴 데이터를 그대로 다시 채점하면 일반화 검증이 아니라 자기 자신을 맞히는
순환 검증이 되어 점수가 항상 높게 나옵니다 (아래 1번 셀에서 이 이유를 코드 주석으로도 설명합니다).

## 이 노트북이 확인하는 것
1. `data/raw`의 500개 중 일부를 hold-out으로 떼어놓고 나머지로만 기준 통계 학습
2. hold-out(안 써본 정상 rep) + `data/raw_test`의 good/bad 전부 채점
3. good 점수 분포 vs bad 점수 분포가 실제로 유의미하게 갈리는지 (Mann-Whitney U 검정)
4. bad 사례마다 `top_issues`가 의도한 오류 유형과 실제로 일치하는지 눈으로 확인
5. 지금 특징(무릎/팔꿈치 min/max/rom/좌우대칭)으로 원래 못 잡는 오류 유형은 무엇인지 정리

## 0. 환경 설정

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

project_root = Path(".").resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.build_dataset import build_dataset, discover_source_files, process_source_file_with_template
from src.preprocessing.angles import EXERCISE_JOINTS, LEFT_HIP, LEFT_KNEE, LEFT_ANKLE, RIGHT_HIP, RIGHT_KNEE, RIGHT_ANKLE
from src.preprocessing.coordiante_normalization import normalize_landmarks
from src.preprocessing.phase_normalization import normalize_phase
from src.scoring_model.reference_stats import fit_reference, save_reference, load_reference
from src.scoring_model.scorer import score_rep
from src.scoring_model.train import META_COLUMNS, is_feature_column

plt.rcParams["figure.figsize"] = (9, 4)
np.set_printoptions(precision=3, suppress=True)

EXERCISE = "squat"          # 검증할 운동
TARGET_LENGTH = 100
HOLDOUT_RATIO = 0.1          # 500개 중 이 비율만큼 학습에서 제외하고 검증용으로 남김
RANDOM_SEED = 42

## 1. 합성 데이터 준비 (실 데이터 없을 때만)

실제로는 `data/raw/`(500개), `data/raw_test/`(good/bad 20개)에 진짜 .npy가 있으면
아래 셀은 아무것도 만들지 않고 건너뜁니다. 지금은 파이프라인 자체를 미리 검증해보기
위해, 실 데이터가 없을 경우에만 합성 데이터로 대체합니다.

합성 "bad" 예시는 일부러 세 종류로 만듭니다:
- `bad_shallow` : 가동범위 부족 (얕은 스쿼트) → `min_angle_knee`로 잡혀야 함
- `bad_partial` : 탑에서 완전히 안 펴짐 → `max_angle_knee`로 잡혀야 함

In [ ]:
RAW_DIR = project_root / "data" / "raw"
RAW_TEST_DIR = project_root / "data" / "raw_test"
PROCESSED_DIR = project_root / "data" / "processed"
PROCESSED_TEST_DIR = project_root / "data" / "processed_test"
RAW_DIR.mkdir(parents=True, exist_ok=True)
RAW_TEST_DIR.mkdir(parents=True, exist_ok=True)


def make_synthetic_rep(exercise, angle_min, angle_max, asym_offset=0.0, seed=0, T=60):
    """EXERCISE_JOINTS 정의를 그대로 이용해, 지정한 각도 범위로 굽혀지는 합성 rep 생성.
    asym_offset을 주면 오른쪽 각도만 그만큼 밀어서 좌우 비대칭을 만든다."""
    rng = np.random.default_rng(seed)
    t = np.linspace(0, np.pi, T)
    curve_l = angle_max - (angle_max - angle_min) * np.sin(t)
    curve_r = curve_l + asym_offset

    joint_name = next(iter(EXERCISE_JOINTS[exercise]))
    a_l, b_l, c_l = EXERCISE_JOINTS[exercise][joint_name]["left"]
    a_r, b_r, c_r = EXERCISE_JOINTS[exercise][joint_name]["right"]

    seg_len = 0.20
    kp = np.zeros((T, 33, 3), dtype=np.float32)
    kp[..., 2] = 0.95
    b_pos_l, b_pos_r = np.array([0.45, 0.5]), np.array([0.55, 0.5])
    c_dir = np.array([0.0, 1.0])

    for i in range(T):
        for side, (a, b, c, curve, b_pos) in [
            ("l", (a_l, b_l, c_l, curve_l, b_pos_l)),
            ("r", (a_r, b_r, c_r, curve_r, b_pos_r)),
        ]:
            theta = np.radians(curve[i]) + rng.normal(0, 0.5 * np.pi / 180)
            sign = -1 if side == "l" else 1
            R = np.array([[np.cos(theta), sign * -np.sin(theta)], [sign * np.sin(theta), np.cos(theta)]])
            a_dir = R @ c_dir
            kp[i, c, :2] = b_pos + c_dir * seg_len
            kp[i, a, :2] = b_pos + a_dir * seg_len
            kp[i, b, :2] = b_pos

    return kp


if not list(RAW_DIR.glob("*.npy")):
    print("data/raw 비어있음 -> 합성 기준 데이터 60개 생성 (실 500개 대신 축소 버전)")
    for i in range(60):
        kp = make_synthetic_rep(EXERCISE, angle_min=80 + np.random.default_rng(i).normal(0, 2),
                                 angle_max=170 + np.random.default_rng(i + 1000).normal(0, 2), seed=i)
        np.save(RAW_DIR / f"{EXERCISE}_ref{i:03d}_keypoints.npy", kp)

if not list(RAW_TEST_DIR.glob("*.npy")):
    print("data/raw_test 비어있음 -> 합성 good/bad 예시 생성")
    for i in range(10):
        kp = make_synthetic_rep(EXERCISE, angle_min=80, angle_max=170, seed=5000 + i)
        np.save(RAW_TEST_DIR / f"{EXERCISE}_good{i:02d}_keypoints.npy", kp)
    for i in range(4):
        kp = make_synthetic_rep(EXERCISE, angle_min=140, angle_max=170, seed=6000 + i)  # 가동범위 부족
        np.save(RAW_TEST_DIR / f"{EXERCISE}_badshallow{i:02d}_keypoints.npy", kp)
    # 참고: 예전엔 asym_offset으로 좌우 비대칭 bad 예시도 만들었으나, select_reliable_side
    # 도입으로 좌우 중 visibility 높은 한쪽만 특징에 쓰는 구조가 되면서 좌우 비대칭 자체가
    # 더 이상 특징에 안 잡힌다 (rom_symmetry 특징이 사라짐). 그래서 이 카테고리는 뺐다.
    for i in range(3):
        kp = make_synthetic_rep(EXERCISE, angle_min=80, angle_max=130, seed=8000 + i)  # 탑에서 안 펴짐
        np.save(RAW_TEST_DIR / f"{EXERCISE}_badpartial{i:02d}_keypoints.npy", kp)

print(f"data/raw: {len(list(RAW_DIR.glob('*.npy')))}개, data/raw_test: {len(list(RAW_TEST_DIR.glob('*.npy')))}개")

## 2. 기준(reference) 500개 전처리 -> rep_features.csv

DTW 위상 정규화를 쓰므로, 이 단계에서 `data/raw`(기준 500개)로 만든 **템플릿(평균 궤적)**을
그대로 저장해뒀다가, 4번 단계에서 `data/raw_test`(good/bad)를 처리할 때도 재사용합니다
(학습/검증에 서로 다른 템플릿을 쓰면 기준 자체가 어긋나므로 반드시 같은 템플릿을 써야 합니다).

In [ ]:
build_dataset(raw_dir=RAW_DIR, output_dir=PROCESSED_DIR, target_length=TARGET_LENGTH)

rep_features_df = pd.read_csv(PROCESSED_DIR / "rep_features.csv")
ref_df = rep_features_df[rep_features_df["exercise"] == EXERCISE].reset_index(drop=True)
feature_names = [c for c in ref_df.columns if is_feature_column(c)]
print(f"\n기준 rep {len(ref_df)}개, 특징 {len(feature_names)}개: {feature_names}")

# build_dataset()이 저장해둔 템플릿을 불러온다 (4번 단계에서 재사용)
with np.load(PROCESSED_DIR / "templates.npz") as _templates:
    REFERENCE_TEMPLATE = _templates[EXERCISE].copy()
print(f"'{EXERCISE}' 템플릿 로드 완료 (길이 {len(REFERENCE_TEMPLATE)})")

## 3. Hold-out 분리 후 기준 통계 학습

500개(또는 합성 데이터 규모) 전부로 학습하지 않고, `HOLDOUT_RATIO`만큼 떼어놓습니다.
이 hold-out은 "학습에 안 쓴 정상 rep"으로서, 뒤에서 `data/raw_test`의 good 예시와
같은 성격의 검증 데이터로 함께 씁니다.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
shuffled_idx = rng.permutation(len(ref_df))
n_holdout = max(1, int(len(ref_df) * HOLDOUT_RATIO))
holdout_idx = shuffled_idx[:n_holdout]
train_idx = shuffled_idx[n_holdout:]

train_matrix = ref_df.loc[train_idx, feature_names].to_numpy()
holdout_matrix = ref_df.loc[holdout_idx, feature_names].to_numpy()

print(f"학습에 사용: {len(train_idx)}개 / hold-out(검증용): {len(holdout_idx)}개")

stats = fit_reference(train_matrix, feature_names, exercise=EXERCISE)
print(f"기준 거리 분포(학습 {len(train_idx)}개 기준): "
      f"min={stats.reference_distances.min():.3f}, median={np.median(stats.reference_distances):.3f}, "
      f"max={stats.reference_distances.max():.3f}")

## 4. `data/raw_test`(good/bad 20개) 전처리 -> rep_features_test.csv

**여기서 `build_dataset()`을 독립적으로 다시 호출하면 안 됩니다** — DTW는 "모범 사례
평균 궤적(템플릿)"에 맞춰 정렬하는 방식이라, `data/raw_test`(good+bad 섞인 소량 데이터)로
템플릿을 따로 만들면 (1) bad 예시가 섞여 템플릿 자체가 오염되고 (2) 학습 때 쓴 템플릿과
달라져서 기준이 어긋납니다. 대신 2번 단계에서 저장해둔 `REFERENCE_TEMPLATE`을 그대로
재사용해 각 파일을 `process_source_file_with_template()`로 처리합니다.

In [ ]:
test_sources = discover_source_files(RAW_TEST_DIR)
test_rows = []
for source in test_sources:
    index_rows, _ = process_source_file_with_template(source, REFERENCE_TEMPLATE)
    test_rows.extend(index_rows)

test_df = pd.DataFrame(test_rows)
test_df = test_df[test_df["exercise"] == EXERCISE].reset_index(drop=True)

# video_id가 "good"으로 시작하면 good, 그 외(bad로 시작)는 bad로 라벨링
test_df["label"] = np.where(test_df["video_id"].str.startswith("good"), "good", "bad")
print(test_df[["video_id", "label"]].to_string(index=False))

## 5. 채점: hold-out(정상) + good + bad 전부

세 그룹 모두 **`stats`(train_idx로만 학습한 통계)**로 채점합니다 — 어느 것도
학습에 쓰인 데이터가 아니므로 전부 공정한 검증입니다.

In [ ]:
holdout_results = [score_rep(holdout_matrix[i], stats) for i in range(len(holdout_matrix))]
test_results = [score_rep(test_df.loc[i, feature_names].to_numpy().astype(float), stats) for i in range(len(test_df))]

holdout_scores = np.array([r["score"] for r in holdout_results])
test_df["score"] = [r["score"] for r in test_results]
test_df["top_issues"] = [r["top_issues"] for r in test_results]

good_scores = test_df.loc[test_df["label"] == "good", "score"].to_numpy()
bad_scores = test_df.loc[test_df["label"] == "bad", "score"].to_numpy()

print(f"Hold-out(정상, 학습 안 씀) 점수: {holdout_scores}")
print(f"\nGood(신규 촬영) 점수: {good_scores}")
print(f"Bad(의도적 오류) 점수: {bad_scores}")

## 6. 분포 시각화 + 통계적 유의성 검정 (Mann-Whitney U)

In [ ]:
all_good = np.concatenate([holdout_scores, good_scores])

plt.boxplot([all_good, bad_scores], tick_labels=["Good (hold-out + 신규)", "Bad (의도적 오류)"])
plt.ylabel("score")
plt.title(f"{EXERCISE}: Good vs Bad 점수 분포")
plt.show()

if len(bad_scores) > 0 and len(all_good) > 0:
    stat, p_value = mannwhitneyu(all_good, bad_scores, alternative="greater")
    print(f"Mann-Whitney U 검정: U={stat:.1f}, p-value={p_value:.4f}")
    if p_value < 0.05:
        print("-> Good 그룹이 Bad 그룹보다 통계적으로 유의미하게 점수가 높습니다 (p<0.05).")
    else:
        print("-> 지금 표본으로는 두 그룹 차이가 통계적으로 유의미하지 않습니다. "
              "표본을 늘리거나 bad 예시의 오류가 지금 특징(min/max/rom/좌우대칭) 범위 안에 있는지 확인하세요.")

print(f"\nGood 최소값: {all_good.min():.1f} / Bad 최대값: {bad_scores.max():.1f}"
      f" ({'분리됨 (겹치지 않음)' if all_good.min() > bad_scores.max() else '일부 겹침'})")

## 7. Bad 사례별 `top_issues` 확인 (의도한 오류 유형과 실제로 일치하는지)

`data/raw_test`의 파일명에 오류 유형(shallow/partial 등)을 넣어두셨다면,
아래에서 각 bad 예시의 `top_issues`가 그 의도와 실제로 맞는지 눈으로 대조해보세요.

In [ ]:
bad_view = test_df[test_df["label"] == "bad"][["video_id", "score", "top_issues"]]
for _, row in bad_view.iterrows():
    print(f"{row['video_id']:20s} score={row['score']:5.1f}  top_issues={row['top_issues']}")

## 8. 정리: 지금 특징으로 못 잡는 오류 유형 체크리스트

아래는 코드로 확인하는 게 아니라, 결과를 보고 직접 채워보시라고 남겨둔 체크리스트입니다.
Bad 예시 중 `top_issues`가 명확하지 않거나 점수가 안 떨어진 경우, 아래 표에서 원인을
찾아보세요 (지난 대화에서 정리한 지금 파이프라인의 알려진 한계입니다).

| 오류 유형 | 지금 특징으로 탐지 가능? |
|---|---|
| 가동범위 부족 / 탑에서 안 펴짐 | ✅ min/max_angle |
| 좌우 비대칭 | ❌ select_reliable_side가 한쪽만 골라 쓰므로 특징에 안 잡힘 (필요하면 재검토) |
| 무릎 valgus (안쪽으로 모임) | ❌ 2D 시상면 각도만 사용 |
| 속도/템포 문제 | ❌ 위상정규화로 속도 정보 소실 |
| 허리/몸통 기울기 | ❌ 몸통 각도가 특징에 없음 |
